Video Inference with Tracking (YOLOv8)

Description:
Runs segmentation + vehicle tracking on a video using a trained YOLOv8 model.
Designed for demo / visualization purposes (not optimized for production).

Author: <Irina Yatskova>
Project: Rural Safe Parking Detection

In [ ]:
!pip install ultralytics

from ultralytics import YOLO
import cv2
import numpy as np
import os
from google.colab import files

In [ ]:
uploaded = files.upload()

# Automatically get filenames
video_path = None
model_path = None

for name in uploaded.keys():
    if name.endswith(".pt"):
        model_path = name
    elif name.endswith(".mp4") or name.endswith(".MP4"):
        video_path = name

print("Model:", model_path)
print("Video:", video_path)

In [ ]:
model = YOLO(model_path)

output_video = "tracking_output.mp4"

cap = cv2.VideoCapture(video_path)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, fps, (frame_width, frame_height))

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, persist=True)[0]

    # Segmentation
    if results.masks is not None:
        masks = results.masks.data.cpu().numpy()
        classes = results.boxes.cls.cpu().numpy()

        for mask, cls in zip(masks, classes):
            mask = cv2.resize(mask, (frame_width, frame_height)) > 0.5
            overlay = np.zeros_like(frame)

            if int(cls) == 0:
                overlay[mask] = (0, 255, 0)
            elif int(cls) == 1:
                overlay[mask] = (0, 0, 255)

            frame = cv2.addWeighted(frame, 1.0, overlay, 0.4, 0)

    # Tracking (vehicles)
    if results.boxes is not None and results.boxes.id is not None:
        boxes = results.boxes.xyxy.cpu().numpy()
        ids = results.boxes.id.cpu().numpy()
        classes = results.boxes.cls.cpu().numpy()

        for box, track_id, cls in zip(boxes, ids, classes):
            if int(cls) == 2:
                x1, y1, x2, y2 = map(int, box)

                cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(frame,
                            f"Vehicle {int(track_id)}",
                            (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6,
                            (255, 0, 0),
                            2)

    out.write(frame)

cap.release()
out.release()

print("✅ Done:", output_video)

In [ ]:
files.download(output_video)